# Flexible Search via GPU

We update the search algorithm to utilize the new GPU backend for dimension computations. We also improve the initialization of the search space to reduce the time needed to process and filter the search space.

In [46]:
# %load dim_backprop_gpu_only.py
"""Exact GPU based computation of polynomial-network neurovariety dimensions. This module has no SageMath or NumPy backend.   

It requires CuPy and a CUDA-capable NVIDIA GPU. CuPy replaces Numpy and provides GPU-accelerated array operations.

Comment: I used an ASUS Dual GeForce RTX 4070 Super.

The public function keeps the original calling convention of the Sage implementation: compute_dimension(network_widths, network_exponent) so that I can use the previous notebooks.

The returned tuple like before is: (sizes, exponent, ambient_dim, expected_dim, dimension, defect).

The computation is exact over the finite fields listed in ``DEFAULT_PRIMES``. One might want to pick larger primes for large runs.

All samples and all output-coordinate pullbacks are differentiated in one batched GPU computation.  

The final Jacobian rank is computed modulo each prime by GPU-parallel row elimination.
"""

from __future__ import annotations

from math import comb
from typing import Iterator, Sequence

try:
    import cupy as cp
except ImportError as exc:
    raise RuntimeError(
        "This GPU-only module requires CuPy. Install the CuPy package that "
        "matches your CUDA version, for example `pip install cupy-cuda12x`."
    ) from exc


DEFAULT_PRIMES = (100003, 100153)
_INT64_MAX = (1 << 63) - 1


def _require_cuda() -> None:
    """Raise a clear error unless a usable CUDA device is visible."""
    try:
        device_count = int(cp.cuda.runtime.getDeviceCount())
    except cp.cuda.runtime.CUDARuntimeError as exc:
        raise RuntimeError(
            "CuPy is installed, but CUDA could not be initialized. Check the "
            "NVIDIA driver, CUDA/CuPy compatibility, and notebook kernel."
        ) from exc

    if device_count < 1:
        raise RuntimeError("No CUDA-capable GPU is visible to CuPy.")


def gpu_information() -> dict[str, object]:
    """Return basic information about the CUDA device used by this module."""
    _require_cuda()
    device_id = int(cp.cuda.runtime.getDevice())
    properties = cp.cuda.runtime.getDeviceProperties(device_id)
    name = properties["name"]
    if isinstance(name, bytes):
        name = name.decode("utf-8", errors="replace")
    return {
        "device_id": device_id,
        "name": name,
        "device_count": int(cp.cuda.runtime.getDeviceCount()),
        "cupy_version": cp.__version__,
        "cuda_runtime_version": int(cp.cuda.runtime.runtimeGetVersion()),
        "driver_version": int(cp.cuda.runtime.driverGetVersion()),
    }


def _is_prime(value: int) -> bool:
    """Deterministic Miller--Rabin test for unsigned 64-bit integers to check for primality."""
    if value < 2:
        return False

    small_primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
    if value in small_primes:
        return True
    if any(value % prime == 0 for prime in small_primes):
        return False

    odd_part = value - 1
    power_of_two = 0
    while odd_part % 2 == 0:
        power_of_two += 1
        odd_part //= 2

    # Deterministic for every n < 2^64.
    for base in (2, 325, 9375, 28178, 450775, 9780504, 1795265022):
        if base % value == 0:
            continue
        witness = pow(base, odd_part, value)
        if witness in (1, value - 1):
            continue
        for _ in range(power_of_two - 1):
            witness = (witness * witness) % value
            if witness == value - 1:
                break
        else:
            return False

    return True


def _weak_compositions(total: int, length: int) -> Iterator[tuple[int, ...]]:
    """Yield nonnegative ``length``-tuples whose entries sum to ``total``. 
    These will be used to form a unisolvent evaluation set for 
    homogeneous polynomials of total degree ``total`` in 
    ``length``-many variables. Tuples should be generated in lexicographic order."""
    if length == 0:
        if total == 0:
            yield ()
        return
    if length == 1:
        yield (total,)
        return

    for first in range(total + 1):
        for rest in _weak_compositions(total - first, length - 1):
            yield (first,) + rest


def _unisolvent_samples(input_dim: int, degree: int, prime: int):
    r"""Construct the homogeneous interpolation points directly on the GPU.

    On the chart ``x_0 = 1``, homogeneous degree-``degree`` forms become
    polynomials of total degree at most ``degree`` in ``input_dim - 1``
    variables.  The integer simplex is an unisolvent evaluation set when
    ``prime > degree``.
    """
    if input_dim < 1:
        raise ValueError("the input layer must have positive width")
    if degree < 0:
        raise ValueError("degree must be nonnegative")
    if prime <= degree:
        raise ValueError(
            f"prime {prime} must exceed polynomial degree {degree}"
        )

    expected = comb(degree + input_dim - 1, input_dim - 1)
    if input_dim == 1:
        return cp.ones((1, 1), dtype=cp.int64)

    points: list[tuple[int, ...]] = []
    for total in range(degree + 1):
        for alpha in _weak_compositions(total, input_dim - 1):
            points.append((1,) + alpha)

    if len(points) != expected:
        raise RuntimeError(
            f"internal sample-count error: got {len(points)}, expected {expected}"
        )

    return cp.asarray(points, dtype=cp.int64) % prime


def _mod_pow(base, exponent: int, prime: int):
    """Elementwise matrix + mod p exponentiation on the GPU."""
    if exponent < 0:
        raise ValueError("exponent must be nonnegative")

    result = cp.ones_like(base, dtype=cp.int64)
    if exponent == 0:
        return result

    power = base.astype(cp.int64, copy=False) % prime
    remaining = int(exponent)
    while remaining:
        if remaining & 1:
            result = (result * power) % prime
        remaining >>= 1
        if remaining:
            power = (power * power) % prime

    return result


def _check_dot_product_safety(widths: Sequence[int], prime: int) -> None:
    """Prevent signed int64 overflow before a matrix product is reduced."""
    largest_inner_dimension = max(int(width) for width in widths)
    worst_case = largest_inner_dimension * (prime - 1) ** 2
    if worst_case > _INT64_MAX:
        raise OverflowError(
            "An int64 GPU matrix product may overflow before reduction modulo "
            "the prime. Use a smaller prime or narrower layers."
        )


def _random_weights(
    widths: Sequence[int], prime: int, seed: int
) -> list[cp.ndarray]:
    """Generate all network weight matrices directly in GPU memory.
    The random number generator is seeded for reproducibility. 
    The generation is uniform over ``0, 1, ..., prime - 1``.
    """
    rng = cp.random.RandomState(seed)
    return [
        rng.randint(
            0,
            prime,
            size=(out_width, in_width),
            dtype=cp.int64,
        )
        for in_width, out_width in zip(widths[:-1], widths[1:])
    ]


def _parameter_offsets(
    weights: Sequence[cp.ndarray],
) -> tuple[list[int], int]:
    offsets: list[int] = []
    total = 0
    for weight in weights:
        offsets.append(total)
        total += int(weight.shape[0] * weight.shape[1])
    return offsets, total


def _batched_weight_jacobian(
    weights: Sequence[cp.ndarray],
    samples: cp.ndarray,
    exponent: int,
    prime: int,
) -> cp.ndarray:
    """Evaluate every output/weight derivative at every sample on the GPU.

    The result has shape ``(number_of_samples, output_width, num_parameters)``.
    Weight matrices are flattened layer by layer in row-major order, matching
    the ordering used by Sage's matrix ``list()`` method in the old code.
    """
    if exponent < 1:
        raise ValueError("network exponent must be at least 1")
    if not weights:
        raise ValueError("the network must contain at least one weight layer")

    activations = [samples]
    preactivations: list[cp.ndarray] = []
    activation = samples

    # Hidden layers use z -> z^exponent; the final layer is linear.
    for weight in weights[:-1]:
        preactivation = cp.matmul(activation, weight.T) % prime
        preactivations.append(preactivation)
        activation = _mod_pow(preactivation, exponent, prime)
        activations.append(activation)

    batch_size = int(samples.shape[0])
    output_width = int(weights[-1].shape[0])
    offsets, num_parameters = _parameter_offsets(weights)

    jacobian = cp.zeros(
        (batch_size, output_width, num_parameters), dtype=cp.int64
    )

    # Final linear layer.
    final_input = activations[-1]
    final_offset = offsets[-1]
    final_input_width = int(weights[-1].shape[1])
    for output_index in range(output_width):
        start = final_offset + output_index * final_input_width
        stop = start + final_input_width
        jacobian[:, output_index, start:stop] = final_input

    if len(weights) == 1:
        return jacobian

    # Derivatives of all output coordinates with respect to the last hidden
    # preactivation: shape (sample, output, hidden neuron).
    delta = cp.broadcast_to(
        weights[-1][None, :, :],
        (batch_size, output_width, int(weights[-1].shape[1])),
    ).copy()
    derivative = (
        (exponent % prime)
        * _mod_pow(preactivations[-1], exponent - 1, prime)
    ) % prime
    delta = (delta * derivative[:, None, :]) % prime

    # Hidden layers from last to first.
    for layer_index in range(len(weights) - 2, -1, -1):
        weight = weights[layer_index]
        layer_input = activations[layer_index]

        gradient = (
            delta[:, :, :, None] * layer_input[:, None, None, :]
        ) % prime

        start = offsets[layer_index]
        stop = start + int(weight.shape[0] * weight.shape[1])
        jacobian[:, :, start:stop] = gradient.reshape(
            batch_size, output_width, stop - start
        )

        if layer_index > 0:
            delta = cp.matmul(delta, weight) % prime
            derivative = (
                (exponent % prime)
                * _mod_pow(
                    preactivations[layer_index - 1], exponent - 1, prime
                )
            ) % prime
            delta = (delta * derivative[:, None, :]) % prime

    return jacobian


def _rank_mod_prime_gpu(
    matrix: cp.ndarray,
    prime: int,
    workspace_bytes: int = 512 * 1024**2,
) -> int:
    """Compute exact matrix rank over GF(prime) using GPU row operations.
    
    Question for future: Can we replace this with a predefined CuPy function?
    """
    if matrix.ndim != 2:
        raise ValueError("rank input must be a matrix")
    if workspace_bytes <= 0:
        raise ValueError("workspace_bytes must be positive")

    reduced = matrix.astype(cp.int64, copy=True) % prime

    # Eliminate along the smaller dimension.
    if reduced.shape[1] > reduced.shape[0]:
        reduced = reduced.T.copy()

    nrows, ncols = map(int, reduced.shape)
    pivot_row = 0

    for column in range(ncols):
        if pivot_row == nrows:
            break

        nonzero = reduced[pivot_row:, column] != 0
        if not bool(cp.any(nonzero).item()):
            continue

        pivot = pivot_row + int(cp.argmax(nonzero).item())
        if pivot != pivot_row:
            temporary = reduced[pivot_row, :].copy()
            reduced[pivot_row, :] = reduced[pivot, :]
            reduced[pivot, :] = temporary

        pivot_value = int(reduced[pivot_row, column].item())
        inverse = pow(pivot_value, prime - 2, prime)
        reduced[pivot_row, column:] = (
            reduced[pivot_row, column:] * inverse
        ) % prime

        # The row updates are parallel CUDA kernels. Chunking bounds temporary
        # memory usage for large Jacobians.
        remaining_columns = ncols - column
        bytes_per_row = max(1, 3 * remaining_columns * 8)
        rows_per_chunk = max(1, workspace_bytes // bytes_per_row)

        start = pivot_row + 1
        while start < nrows:
            stop = min(nrows, start + rows_per_chunk)
            factors = reduced[start:stop, column].copy()
            reduced[start:stop, column:] = (
                reduced[start:stop, column:]
                - factors[:, None] * reduced[pivot_row, column:][None, :]
            ) % prime
            start = stop

        pivot_row += 1

    return pivot_row


def compute_dimension(
    network_widths: Sequence[int],
    network_exponent: int,
    *,
    primes: Sequence[int] = DEFAULT_PRIMES,
    seed: int = 20260630,
    rank_workspace_bytes: int = 512 * 1024**2,
    verbose: bool = False,
):
    """Compute the neurovariety dimension entirely with the CUDA backend.

    Parameters
    ----------
    network_widths:
        Layer widths ``[d0, d1, ..., dL]``.
    network_exponent:
        Common hidden-layer activation exponent.
    primes:
        Prime moduli used to cross-check the generic rank.
    seed:
        Base random seed for the network weights.
    rank_workspace_bytes:
        Approximate upper bound for temporary elimination workspace.
    verbose:
        Print GPU and rank information.

    Returns
    -------
    tuple
        ``(sizes, exponent, ambient_dim, expected_dim, dimension, defect)``.
    """
    _require_cuda()

    widths = tuple(int(width) for width in network_widths)
    exponent = int(network_exponent)

    if len(widths) < 2:
        raise ValueError("network_widths must contain input and output widths")
    if any(width <= 0 for width in widths):
        raise ValueError("all network widths must be positive")
    if exponent < 1:
        raise ValueError("network_exponent must be at least 1")
    if not primes:
        raise ValueError("at least one prime is required")

    degree = exponent ** (len(widths) - 2)
    ambient_per_output = comb(degree + widths[0] - 1, widths[0] - 1)
    ambient_dim = ambient_per_output * widths[-1]
    num_parameters = sum(
        in_width * out_width
        for in_width, out_width in zip(widths[:-1], widths[1:])
    )

    dimensions: list[int] = []

    if verbose:
        info = gpu_information()
        print(
            f"GPU {info['device_id']}: {info['name']} | "
            f"CuPy {info['cupy_version']} | "
            f"CUDA runtime {info['cuda_runtime_version']}"
        )

    for prime_index, prime_value in enumerate(primes):
        prime = int(prime_value)
        if not _is_prime(prime):
            raise ValueError(f"modulus {prime} is not prime")
        _check_dot_product_safety(widths, prime)

        samples = _unisolvent_samples(widths[0], degree, prime)
        weights = _random_weights(
            widths,
            prime,
            seed + 1_000_003 * prime_index + prime,
        )

        jacobian = _batched_weight_jacobian(
            weights,
            samples,
            exponent,
            prime,
        )
        rank_matrix = jacobian.reshape(
            ambient_per_output * widths[-1], num_parameters
        )
        dimension = _rank_mod_prime_gpu(
            rank_matrix,
            prime,
            workspace_bytes=rank_workspace_bytes,
        )
        dimensions.append(dimension)

        # Ensure kernels for this prime have completed before reporting and
        # releasing memory.
        cp.cuda.Stream.null.synchronize()

        if verbose:
            print(
                f"prime={prime}, samples={ambient_per_output}, "
                f"rank_matrix={tuple(rank_matrix.shape)}, rank={dimension}"
            )

        del rank_matrix, jacobian, weights, samples
        cp.get_default_memory_pool().free_all_blocks()

    if not all(dimension == dimensions[0] for dimension in dimensions):
        raise ValueError(
            "different dimensions over finite fields: " + str(dimensions)
        )

    naive_bound = sum(
        (in_width - 1) * out_width
        for in_width, out_width in zip(widths[:-1], widths[1:])
    ) + widths[-1]
    expected_dim = min(ambient_dim, naive_bound)
    dimension = dimensions[0]

    return (
        list(widths),
        exponent,
        ambient_dim,
        expected_dim,
        dimension,
        expected_dim - dimension,
    )


# Search Algorithm

In [47]:
import os
import ast
import math
import itertools
import random as py_random
import pandas as pd
from tqdm import tqdm

In [56]:
# Helper Functions

def calculate_parameter_count(hidden: tuple, d_0: int, d_h: int) -> int:
    """Calculates the total number of weights and biases in the network."""
    sizes = [d_0] + list(hidden) + [d_h]
    return sum(m * n for m, n in zip(sizes[:-1], sizes[1:]))

def is_less_or_equal(t1: tuple, t2: tuple) -> bool:
    """Returns True if every element in t1 is <= the corresponding element in t2."""
    if len(t1) != len(t2):
        return False
    return all(a <= b for a, b in zip(t1, t2))

def evaluate_single_architecture(hidden_tuple: tuple, h: int, d_0: int, d_h: int, exponent: int):
    """Worker function to evaluate a single architecture and format the result.
    This uses the previous compute_dimension function.
    """
    sizes = [d_0] + list(hidden_tuple) + [d_h]
    arch_str = str(sizes)
    
    try:
        _, _, amb, _, dim, _ = compute_dimension(sizes, exponent)
        params = calculate_parameter_count(hidden_tuple, d_0, d_h)
        is_full = (dim == amb)
        
        status = "FULL " if is_full else "SHORT"
        print(f"  [{status}] {arch_str} -> Rank: {dim}/{amb} (Params: {params})")
        
        return {
            "h": h,
            "exponent": exponent,
            "architecture": arch_str,
            "num_parameters": params,
            "dimension_computed": int(dim),
            "ambient_dimension": int(amb),
            "is_full_dimension": is_full,
            "is_minimal": False 
        }
    except Exception as e:
        print(f"  [ERROR] {arch_str} failed: {e}")
        return None

In [49]:
# Core Search & Pruning Algorithm

def parameter_boundary_search(
    h_values: list, max_width=6, min_width=1, exponent=2, 
    d_0=2, d_h=1, csv_filename="architecture_search_log.csv", 
    user_guesses=None, layer_bounds=None
):

    # Load or initialize Database
    if os.path.exists(csv_filename):
        print(f"Loading existing database from '{csv_filename}'...")
        df = pd.read_csv(csv_filename)
    else:
        print("No existing database found. Starting fresh...")
        df = pd.DataFrame(columns=[
            "h", "exponent", "architecture", "num_parameters", 
            "dimension_computed", "ambient_dimension", "is_full_dimension", "is_minimal"
        ])

    evaluated_architectures = set(df["architecture"].tolist())
    new_records = []

    for h in h_values:
        print(f"\n--- RANDOM SEARCH: h={h}, exponent={exponent} ---")
        
        num_hidden = h - 1
        degree = exponent ** num_hidden
        ambient_dim = math.comb(degree + d_0 - 1, d_0 - 1) * d_h
        print(f"  Target Ambient Dimension: {ambient_dim}")

        # Extract known boundaries
        known_minimal = set()
        known_short = set()
        
        if not df.empty:
            subset_df = df[(df["h"] == h) & (df["exponent"] == exponent)]
            
            min_df = subset_df[subset_df["is_minimal"] == True]
            known_minimal.update(tuple(ast.literal_eval(a)[1:-1]) for a in min_df["architecture"])
            
            short_df = subset_df[subset_df["is_full_dimension"] == False]
            known_short.update(tuple(ast.literal_eval(a)[1:-1]) for a in short_df["architecture"])

        # 1. Evaluate User Guesses
        if user_guesses and h in user_guesses:
            for guess in user_guesses[h]:
                arch_str = str([d_0] + list(guess) + [d_h])
                if arch_str in evaluated_architectures:
                    continue
                
                print(f"  Evaluating Guess: {guess}...")
                res = evaluate_single_architecture(guess, h, d_0, d_h, exponent)
                if not res: continue
                
                evaluated_architectures.add(res["architecture"])
                new_records.append(res)
                
                if res["is_full_dimension"]:
                    known_minimal.add(guess)
                else:
                    known_short.add(guess)

# 2. Build and Filter Search Pool
        print("  Generating candidate pool...")
        
        # Builds the Search Pool but imposing bounds on the individual layers

        if layer_bounds and h in layer_bounds:
            bounds = layer_bounds[h]
            if len(bounds) != num_hidden:
                print(f"  [WARNING] layer_bounds for h={h} has length {len(bounds)}, but expected {num_hidden} hidden layers. Skipping this depth...")
                continue
            
            print(f"  Using custom per-layer bounds: {bounds}")
            ranges = [range(max(1, b_min), b_max + 1) for b_min, b_max in bounds]
            all_possible = itertools.product(*ranges)
            
            # calculate total items for the progress bar
            total_combinations = math.prod(len(r) for r in ranges)
            
        else:
            print(f"  Using global bounds: min_width={min_width}, max_width={max_width}")
            width_range = range(max(1, min_width), max_width + 1)
            all_possible = itertools.product(width_range, repeat=num_hidden)
            
            # calculate total items for the progress bar
            total_combinations = len(width_range) ** num_hidden
        
        candidate_pool = []
        rejected_count = 0
        
        # wrap in tqdm. useful for long run times.
        for hidden in tqdm(all_possible, total=total_combinations, desc="  Filtering", leave=False, dynamic_ncols=True):
            arch_str = str([d_0] + list(hidden) + [d_h])
            if arch_str in evaluated_architectures:
                continue
                
            params = calculate_parameter_count(hidden, d_0, d_h)
            
            # pruning 
            if (params < ambient_dim or 
                any(is_less_or_equal(m, hidden) for m in known_minimal) or 
                any(is_less_or_equal(hidden, s) for s in known_short)):
                rejected_count += 1
                continue
                
            candidate_pool.append(hidden)
            
        print(f"  Pruned {rejected_count} impossible/redundant architectures.")
        print(f"  Starting sequential evaluation on {len(candidate_pool)} viable candidates.")

        # shuffle queue 
        py_random.shuffle(candidate_pool)

        try:
            while candidate_pool:
                target = candidate_pool.pop(0)
                
                res = evaluate_single_architecture(target, h, d_0, d_h, exponent)
                if not res: continue
                
                evaluated_architectures.add(res["architecture"])
                new_records.append(res)
                
                before_len = len(candidate_pool)
                if res["is_full_dimension"]:
                    known_minimal.add(target)
                    # remove all supersets from the queue
                    candidate_pool = [c for c in candidate_pool if not is_less_or_equal(target, c)]
                    pruned = before_len - len(candidate_pool)
                    if pruned > 0: print(f"    [Upward Pruned] {pruned} supersets removed from queue.")
                else:
                    known_short.add(target)
                    # remove all subsets from the queue
                    candidate_pool = [c for c in candidate_pool if not is_less_or_equal(c, target)]
                    pruned = before_len - len(candidate_pool)
                    if pruned > 0: print(f"    [Downward Pruned] {pruned} subsets removed from queue.")

        except KeyboardInterrupt: # incase one wants to halt a long run early and still save the results
            print("\n[Interrupt] User halted the loop early. Saving progress and continuing execution...")

    # 4. Save and determine minimality based on results in the .csv
    # This does not guarantee the examples are truly minimality -- need to do actual checks elsewhere.
    if new_records:
        new_df = pd.DataFrame(new_records).dropna(axis=1, how='all')
        if not df.empty:
            df = pd.concat([df, new_df], ignore_index=True)
        else:
            df = new_df
        print(f"\nAdded {len(new_records)} new architectures to the database.")

    print("Re-evaluating minimal filling properties (grouped by depth h and exponent)...")
    if not df.empty:
        df["is_minimal"] = False 
        
        # calculate if minimal 
        # this does not actual check minimality fully -- just minimality base on search so far
        # see verify_minimal.ipynb

        for (h_val, exp_val), group in df[df["is_full_dimension"] == True].groupby(["h", "exponent"]):
            full_archs = [ast.literal_eval(arch) for arch in group["architecture"]]
            
            for idx, row in group.iterrows():
                parsed_config = ast.literal_eval(row["architecture"])
                # it is minimal if no other full architecture is strictly smaller than it
                is_min = not any(
                    other != parsed_config and is_less_or_equal(other, parsed_config) 
                    for other in full_archs
                )
                df.at[idx, "is_minimal"] = is_min

        df.to_csv(csv_filename, index=False)
        print(f"Database successfully saved to '{csv_filename}'.\n")
    return df

# Executing the Search

In [51]:
# This adjusts which depths you want to consider
h_values_to_test = [2,3,4,5] 


d0 = 2
dh = 1
r = 5

# Optional: provide guess as tuple for the hidden layers
my_guesses = {} 


# Provide individual bounds for each hidden layer (length must match h - 1)
# Format: { h_value: [(min1, max1), (min2, max2), ...] }

custom_bounds = {
    5: [
        (1, min(dh*(r**d0), math.comb(r**(5-1)+d0-1, r**(5-1))) ),  # d1
        (1, min(dh*(r** (2*d0)), math.comb(r**(5-2)+d0-1, r**(5-2))) ),  # d2
        (1, min(dh*(r** (3*d0)), math.comb(r**(5-3)+d0-1, r**(5-3))) ),  # d3
        (1, min(dh*(r** (4*d0)), math.comb(r**(5-4)+d0-1, r**(5-4))) ), # d4
    ],

    6: [
        (1, min(dh*(r**d0), math.comb(r**(6-1)+d0-1, r**(6-1))) ),  # d1
        (1, min(dh*(r** (2*d0)), math.comb(r**(6-2)+d0-1, r**(6-2))) ),  # d2
        (1, min(dh*(r** (3*d0)), math.comb(r**(6-3)+d0-1, r**(6-3))) ),  # d3
        (1, min(dh*(r** (4*d0)), math.comb(r**(6-4)+d0-1, r**(6-4))) ), # d4
        (1, min(dh*(r** (5*d0)), math.comb(r**(6-5)+d0-1, r**(6-5))) ), # d5
    ],
}

# This is the KTB threshold.

# def bound(L, i, r, d0, dh):
#     return (1, min(dh * r**(i*d0), math.comb(r**(L-i) + d0 - 1, r**(L-i))))

# custom_bounds = {
#     L: [bound(L, i, r, d0, dh) for i in range(1, L)]
#     for L in range(1, 9)  # depths 1 through 8
# }

# Run the bounded frontier search

df_results = parameter_boundary_search(
    h_values=h_values_to_test, 
    max_width=40,        # Fallback if `h` not in layer_bounds
    min_width=1,         # Fallback if `h` not in layer_bounds
    exponent=r,          
    d_0=d0,              
    d_h=dh,              
    csv_filename=f"../data/raw/{d0}_{dh}_r{r}_architectures.csv",
    user_guesses=my_guesses,
    layer_bounds=custom_bounds  # Injecting the new feature here
)

Loading existing database from '../data/raw/2_1_r5_architectures.csv'...

--- RANDOM SEARCH: h=2, exponent=5 ---
  Target Ambient Dimension: 6
  Generating candidate pool...
  Using global bounds: min_width=1, max_width=40


  Pruned 36 impossible/redundant architectures.
  Starting sequential evaluation on 0 viable candidates.

--- RANDOM SEARCH: h=3, exponent=5 ---
  Target Ambient Dimension: 26
  Generating candidate pool...
  Using global bounds: min_width=1, max_width=40


  Pruned 1552 impossible/redundant architectures.
  Starting sequential evaluation on 0 viable candidates.

--- RANDOM SEARCH: h=4, exponent=5 ---
  Target Ambient Dimension: 126
  Generating candidate pool...
  Using global bounds: min_width=1, max_width=40


  Pruned 62520 impossible/redundant architectures.
  Starting sequential evaluation on 0 viable candidates.

--- RANDOM SEARCH: h=5, exponent=5 ---
  Target Ambient Dimension: 626
  Generating candidate pool...
  Using custom per-layer bounds: [(1, 25), (1, 126), (1, 26), (1, 6)]


  Pruned 178610 impossible/redundant architectures.
  Starting sequential evaluation on 308733 viable candidates.
  [SHORT] [2, 13, 124, 20, 4, 1] -> Rank: 504/626 (Params: 4202)
    [Downward Pruned] 77885 subsets removed from queue.
  [SHORT] [2, 10, 104, 6, 5, 1] -> Rank: 180/626 (Params: 1719)
    [Downward Pruned] 2031 subsets removed from queue.
  [SHORT] [2, 13, 65, 11, 5, 1] -> Rank: 330/626 (Params: 1646)
    [Downward Pruned] 1831 subsets removed from queue.
  [SHORT] [2, 19, 108, 23, 2, 1] -> Rank: 252/626 (Params: 4622)
    [Downward Pruned] 24071 subsets removed from queue.
  [SHORT] [2, 14, 51, 12, 4, 1] -> Rank: 348/626 (Params: 1406)
    [Downward Pruned] 263 subsets removed from queue.
  [SHORT] [2, 1, 117, 23, 2, 1] -> Rank: 2/626 (Params: 2858)
    [Downward Pruned] 53 subsets removed from queue.
  [SHORT] [2, 8, 111, 5, 5, 1] -> Rank: 150/626 (Params: 1489)
    [Downward Pruned] 209 subsets removed from queue.
  [SHORT] [2, 23, 110, 11, 1, 1] -> Rank: 126/626 (Param

In [52]:
# # Looped version for searches

# for d0 in range(1,10):
#     for dh in range(1,10):
#         for r in range(1,10):
#             h_values_to_test = [2,3] 
#             my_guesses = {}
#             # Format: { h_value: [(min1, max1), (min2, max2), ...] }
#             custom_bounds = {
#                 6: [
#                     (3, 4),  # d1
#                     (3, 10),  # d2
#                     (5, 28),  # d3
#                     (9, 82), # d4
#                     (6, 18)  # d5
#                 ],
#             }

#             df_results = parameter_boundary_search(
#                 h_values=h_values_to_test, 
#                 max_width=300,        # Fallback if `h` not in layer_bounds
#                 min_width=1,         # Fallback if `h` not in layer_bounds
#                 exponent=r,          
#                 d_0=d0,              
#                 d_h=dh,              
#                 csv_filename=f"../data/raw/{d0}_{dh}_r{r}_architectures.csv",
#                 user_guesses=my_guesses,
#                 layer_bounds=custom_bounds  # Injecting the new feature here
#             )

In [53]:
# Display the minimal architectures found so far
print("\n=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===")
if not df_results.empty:
    minimal_archs = df_results[df_results['is_minimal'] == True]
    
    if not minimal_archs.empty:
        # Sort by parameters for easier reading
        minimal_archs = minimal_archs.sort_values(by="num_parameters")
        print(minimal_archs[["architecture", "num_parameters", "dimension_computed"]].to_string(index=False))
    else:
        print("No minimal full architectures found matching the criteria.")
else:
    print("Database is empty.")


=== CURRENT MINIMAL FILLING ARCHITECTURES IN DATABASE ===
          architecture  num_parameters  dimension_computed
             [2, 3, 1]             9.0                 6.0
          [2, 4, 6, 1]            38.0                26.0
          [2, 5, 5, 1]            40.0                26.0
      [2, 5, 11, 7, 1]           149.0               126.0
      [2, 4, 9, 11, 1]           154.0               126.0
      [2, 4, 14, 6, 1]           154.0               126.0
      [2, 5, 14, 5, 1]           155.0               126.0
      [2, 5, 9, 10, 1]           155.0               126.0
       [2, 6, 9, 9, 1]           156.0               126.0
      [2, 4, 7, 15, 1]           156.0               126.0
      [2, 4, 16, 5, 1]           157.0               126.0
      [2, 4, 8, 13, 1]           157.0               126.0
      [2, 5, 7, 14, 1]           157.0               126.0
     [2, 4, 10, 10, 1]           158.0               126.0
      [2, 4, 6, 18, 1]           158.0               126

# Results of the Search

In [54]:
# %load is_unimodal.py

import ast

def is_unimodal(data):
    # 1. Parse the string into a list safely
    if isinstance(data, str):
        try:
            # ast.literal_eval safely evaluates strings containing Python literals
            data = ast.literal_eval(data)
        except (ValueError, SyntaxError):
            raise ValueError(f"Could not parse the string: '{data}'. Ensure it is formatted like '[1, 2, 3]'.")
            
    # 2. Validate the data type
    if not isinstance(data, list):
        raise TypeError("Input must be a list or a string representation of a list.")
        
    # 3. Core unimodal logic
    n = len(data)
    if n <= 2:
        return True
        
    i = 0
    
    # Phase 1: Walk up the non-decreasing slope
    while i + 1 < n and data[i] <= data[i + 1]:
        i += 1
        
    # Phase 2: Walk down the non-increasing slope
    while i + 1 < n and data[i] >= data[i + 1]:
        i += 1
        
    # Phase 3: Check if we reached the end
    return i == n - 1

In [55]:
USER_DEPTH = 5
d0=3
dL=1
r=3

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_full_dimension'] == True)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_full_dimension'] == True)]))

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED POTENTIAL MINIMAL FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True)]))

df = pd.read_csv(f'../data/raw/{d0}_{dL}_r{r}_architectures.csv')
df['is_unimodal'] = df['architecture'].apply(is_unimodal)
print("--"*15 + "DISCOVERED POTENTIAL NONUNIMODAL MINIMAL FILLING ARCHITECTURES" + "--"*15)
display(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True) & (df['is_unimodal']==False)])
print(len(df[(df['h']==USER_DEPTH) & (df['is_minimal'] == True) & (df['is_unimodal']==False)]))

------------------------------DISCOVERED FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
635,5,3,"[3, 21, 14, 55, 47, 1]",3759,3403,3403,True,False,False
637,5,3,"[3, 20, 47, 46, 50, 1]",5512,3403,3403,True,False,False
640,5,3,"[3, 11, 44, 55, 48, 1]",5625,3403,3403,True,False,True
641,5,3,"[3, 42, 41, 53, 30, 1]",5641,3403,3403,True,False,False
642,5,3,"[3, 21, 49, 55, 42, 1]",6139,3403,3403,True,False,True
...,...,...,...,...,...,...,...,...,...
12247,5,3,"[3, 6, 48, 53, 13, 1]",3552,3403,3403,True,True,True
12250,5,3,"[3, 8, 45, 33, 51, 1]",3603,3403,3403,True,True,False
12251,5,3,"[3, 8, 33, 47, 36, 1]",3567,3403,3403,True,True,True
12255,5,3,"[3, 6, 35, 46, 37, 1]",3577,3403,3403,True,True,True


5508
------------------------------DISCOVERED POTENTIAL MINIMAL FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
1345,5,3,"[3, 8, 37, 42, 39, 1]",3551,3403,3403,True,True,True
1442,5,3,"[3, 9, 37, 51, 25, 1]",3547,3403,3403,True,True,True
1625,5,3,"[3, 7, 43, 35, 48, 1]",3555,3403,3403,True,True,False
1784,5,3,"[3, 9, 38, 50, 25, 1]",3544,3403,3403,True,True,True
1962,5,3,"[3, 8, 36, 41, 42, 1]",3552,3403,3403,True,True,True
...,...,...,...,...,...,...,...,...,...
12247,5,3,"[3, 6, 48, 53, 13, 1]",3552,3403,3403,True,True,True
12250,5,3,"[3, 8, 45, 33, 51, 1]",3603,3403,3403,True,True,False
12251,5,3,"[3, 8, 33, 47, 36, 1]",3567,3403,3403,True,True,True
12255,5,3,"[3, 6, 35, 46, 37, 1]",3577,3403,3403,True,True,True


1630
------------------------------DISCOVERED POTENTIAL NONUNIMODAL MINIMAL FILLING ARCHITECTURES------------------------------


,h,exponent,architecture,num_parameters,dimension_computed,ambient_dimension,is_full_dimension,is_minimal,is_unimodal
1625,5,3,"[3, 7, 43, 35, 48, 1]",3555,3403,3403,True,True,False
2575,5,3,"[3, 7, 45, 37, 42, 1]",3597,3403,3403,True,True,False
2963,5,3,"[3, 6, 47, 35, 45, 1]",3565,3403,3403,True,True,False
3027,5,3,"[3, 6, 46, 35, 47, 1]",3596,3403,3403,True,True,False
3097,5,3,"[3, 15, 42, 36, 44, 1]",3815,3403,3403,True,True,False
...,...,...,...,...,...,...,...,...,...
12198,5,3,"[3, 13, 9, 54, 54, 1]",3612,3403,3403,True,True,False
12215,5,3,"[3, 8, 39, 37, 47, 1]",3565,3403,3403,True,True,False
12231,5,3,"[3, 10, 44, 36, 41, 1]",3571,3403,3403,True,True,False
12239,5,3,"[3, 13, 37, 34, 55, 1]",3703,3403,3403,True,True,False


195
